In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import os
from glob import glob

plt.rcParams.update({
    "font.size": 14,
    "axes.titlesize": 16,
    "axes.labelsize": 16,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 14,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "lines.linewidth": 2,
    "lines.markersize": 6,
    "figure.figsize": (5, 4),
})


In [ ]:
# ==========================================
# 1. Helpers for postprocessed AN/node spectra
# ==========================================

base_dir = "./"

def discover_temperatures(base_dir="./"):
    summary = os.path.join(base_dir, "summary_all.csv")
    if os.path.exists(summary):
        df = pd.read_csv(summary)
        return np.sort(df['T'].dropna().unique())
    vals = []
    for path in glob(os.path.join(base_dir, "T_*")):
        try:
            vals.append(float(os.path.basename(path).split("_", 1)[1]))
        except ValueError:
            pass
    return np.array(sorted(vals))

def find_T_dir(T, base_dir="./"):
    candidates = [f"T_{T}", f"T_{T:g}", f"T_{T:.3f}", f"T_{T:.2f}"]
    for name in candidates:
        path = os.path.join(base_dir, name)
        if os.path.isdir(path):
            return path
    return os.path.join(base_dir, f"T_{T:g}")

def read_omega0_series(data_dir, filename, value_col, err_col='Error'):
    file_path = os.path.join(data_dir, filename)
    if not os.path.exists(file_path):
        print(f"Warning: File not found {file_path}")
        return np.nan, np.nan, np.nan
    df = pd.read_csv(file_path).sort_values('omega')
    idx0 = int(np.argmin(np.abs(df['omega'].to_numpy())))
    err = df[err_col].iloc[idx0] if err_col in df.columns else np.nan
    return df[value_col].iloc[idx0], err, df['omega'].iloc[idx0]

def read_series(data_dir, filename, value_col):
    file_path = os.path.join(data_dir, filename)
    if not os.path.exists(file_path):
        return None
    df = pd.read_csv(file_path).sort_values('omega')
    err = df['Error'].to_numpy(dtype=float) if 'Error' in df.columns else None
    return df['omega'].to_numpy(dtype=float), df[value_col].to_numpy(dtype=float), err

def read_peak_summary(data_dir):
    path = os.path.join(data_dir, 'spectra_path_peaks.csv')
    if not os.path.exists(path):
        return pd.DataFrame()
    return pd.read_csv(path)

def symmetrize_series(omega, vals, err=None):
    order = np.argsort(omega)
    omega = np.asarray(omega, dtype=float)[order]
    vals = np.asarray(vals, dtype=float)[order]
    vals_mirror = np.interp(-omega, omega, vals, left=np.nan, right=np.nan)
    vals_sym = 0.5 * (vals + vals_mirror)
    if err is None:
        return omega, vals_sym, None
    err = np.asarray(err, dtype=float)[order]
    err_mirror = np.interp(-omega, omega, err, left=np.nan, right=np.nan)
    err_sym = 0.5 * np.sqrt(err**2 + err_mirror**2)
    return omega, vals_sym, err_sym

def estimate_tc_from_summary(base_dir="./", rho_col='Superfluid_Stiffness_mean'):
    summary = os.path.join(base_dir, "summary_all.csv")
    if not os.path.exists(summary):
        return np.nan
    df = pd.read_csv(summary).sort_values('T')
    if rho_col not in df.columns or len(df) < 2:
        return np.nan
    T = df['T'].to_numpy(dtype=float)
    diff = df[rho_col].to_numpy(dtype=float) - (2.0 / np.pi) * T
    finite = np.isfinite(T) & np.isfinite(diff)
    T = T[finite]
    diff = diff[finite]
    if len(T) < 2:
        return np.nan
    for i in range(len(T) - 1):
        if diff[i] == 0:
            return float(T[i])
        if diff[i] * diff[i + 1] <= 0:
            return float(T[i] - diff[i] * (T[i + 1] - T[i]) / (diff[i + 1] - diff[i]))
    return np.nan

def temperature_norm(T_values):
    T_values = np.asarray(T_values, dtype=float)
    cmap = plt.cm.turbo
    if len(T_values) > 1 and np.nanmax(T_values) > np.nanmin(T_values):
        norm = mcolors.Normalize(vmin=np.nanmin(T_values), vmax=np.nanmax(T_values))
    else:
        t0 = float(T_values[0]) if len(T_values) else 0.0
        norm = mcolors.Normalize(vmin=t0 - 1e-12, vmax=t0 + 1e-12)
    return cmap, norm

def nearest_temperature(T_values, target):
    T_values = np.asarray(T_values, dtype=float)
    if len(T_values) == 0 or not np.isfinite(target):
        return np.nan
    return float(T_values[np.nanargmin(np.abs(T_values - target))])


In [ ]:
# ==========================================
# 2. Main loop
# ==========================================

T_list = discover_temperatures(base_dir)
T_list = T_list[(T_list >= 0.005) & (T_list <= 0.100)]
Tc = estimate_tc_from_summary(base_dir)
T_near_Tc = nearest_temperature(T_list, Tc)

results = {
    "T": [],
    "AN0": [], "AN0_err": [],
    "node0": [], "node0_err": [],
    "AN_kx": [], "AN_ky": [],
    "node_kx": [], "node_ky": [],
}

for T in T_list:
    data_dir = find_T_dir(T, base_dir)
    print(f"Processing {os.path.basename(data_dir)}...", end='\r')
    an0, an_err, _ = read_omega0_series(data_dir, 'spectra_dos_AN.csv', 'DOS_AN')
    nd0, nd_err, _ = read_omega0_series(data_dir, 'spectra_dos_node.csv', 'DOS_node')
    peaks = read_peak_summary(data_dir)

    results['T'].append(T)
    results['AN0'].append(an0); results['AN0_err'].append(an_err)
    results['node0'].append(nd0); results['node0_err'].append(nd_err)

    for kind, prefix in [('AN', 'AN'), ('node', 'node')]:
        sub = peaks[peaks['kind'] == kind] if not peaks.empty and 'kind' in peaks.columns else pd.DataFrame()
        if len(sub) > 0:
            results[f'{prefix}_kx'].append(sub['kx'].mean())
            results[f'{prefix}_ky'].append(sub['ky'].mean())
        else:
            results[f'{prefix}_kx'].append(np.nan)
            results[f'{prefix}_ky'].append(np.nan)

print()
print("Analysis complete.")
if np.isfinite(Tc):
    print(f"Tc from Kubo-BKT crossing: {Tc:.5f}; highlighted T={T_near_Tc:g}")
for key in results:
    results[key] = np.array(results[key])


In [ ]:
# ==========================================
# 3. Symmetrized A(omega) at postprocessed AN/node for different T
# ==========================================

if len(T_list) == 0:
    raise RuntimeError("No temperature points found.")

cmap, norm = temperature_norm(T_list)
fig, axes = plt.subplots(1, 2, figsize=(9.0, 3.6), dpi=300, sharex=True)
plot_specs = [
    (axes[0], 'spectra_dos_AN.csv', 'DOS_AN', r'$A_{\mathrm{AN}}(\omega)$'),
    (axes[1], 'spectra_dos_node.csv', 'DOS_node', r'$A_{\mathrm{node}}(\omega)$'),
]

for T in T_list:
    data_dir = find_T_dir(T, base_dir)
    highlight = np.isfinite(T_near_Tc) and np.isclose(T, T_near_Tc)
    color = 'black' if highlight else cmap(norm(T))
    linewidth = 3.0 if highlight else 1.2
    zorder = 5 if highlight else 2
    label = rf"T={T:g} closest to $T_c$" if highlight else None

    for ax, filename, value_col, ylabel in plot_specs:
        out = read_series(data_dir, filename, value_col)
        if out is None:
            continue
        omega, vals, err = symmetrize_series(*out)
        mask = np.isfinite(omega) & np.isfinite(vals)
        ax.plot(omega[mask], vals[mask], color=color, linewidth=linewidth,
                zorder=zorder, label=label)
        if err is not None:
            mask_err = mask & np.isfinite(err)
            ax.fill_between(omega[mask_err], vals[mask_err] - err[mask_err], vals[mask_err] + err[mask_err],
                            color=color, alpha=0.10 if highlight else 0.08, linewidth=0)
        ax.set_ylabel(ylabel)
        ax.set_xlim(-0.3, 0.3)
        ax.axvline(0, color='gray', linestyle='--', linewidth=1)

for ax in axes:
    ax.set_xlabel(r'$\omega$')

if np.isfinite(T_near_Tc):
    axes[0].legend(frameon=False, fontsize=10)

sm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=axes, pad=0.02)
cbar.set_label(r'$T$')
plt.show()


In [ ]:
df = pd.read_csv("summary_all.csv").sort_values('T')
Tc = estimate_tc_from_summary(base_dir)
Tc


In [ ]:
# fig, ax = plt.subplots(dpi=300)

# ax.errorbar(results["T"], results["val_anti"], yerr=results["err_anti"], 
#             fmt='-o', color='black')

# ax.axvline(x=Tc, color='gray', linestyle=':', linewidth=1.5, label=rf'$T_c$')

# ax.set_xlabel(r'$T$')
# ax.set_ylabel(r'$A(\mathbf{k}=\mathbf{M}, \omega=0)$')
# ax.set_xlim(0,0.08) 
# # ax.set_ylim(0.58,0.76) 
# ax.legend(loc='best', frameon=False)
# plt.show()

In [ ]:
fig, ax = plt.subplots(dpi=300)

ax.errorbar(results["T"], results["AN0"], yerr=results["AN0_err"],
            fmt='-o', color='tab:orange', label=r'AN from M-X path')
ax.errorbar(results["T"], results["node0"], yerr=results["node0_err"],
            fmt='-s', color='tab:green', label=r'node from $3\times3$ patch near $\Gamma$-X path')

if np.isfinite(Tc):
    ax.axvline(x=Tc, color='gray', linestyle=':', linewidth=1.5, label=rf'$T_c={Tc:.4f}$')

ax.set_xlabel(r'$T$')
ax.set_ylabel(r'$A(\omega=0)$')
ax.set_xlim(0, 0.105)
ax.legend(loc='best', frameon=False)
plt.show()


In [ ]:
# ==========================================
# 4. dA_AN(omega=0)/dT
# ==========================================

t_vals = results["T"].copy()
a0_vals = results["AN0"].copy()
mask = np.isfinite(t_vals) & np.isfinite(a0_vals)
t_vals = t_vals[mask]
a0_vals = a0_vals[mask]
sort_idx = np.argsort(t_vals)
t_vals = t_vals[sort_idx]
a0_vals = a0_vals[sort_idx]

if len(t_vals) < 2:
    print("Not enough points to compute dA/dT.")
else:
    dA_dT = np.gradient(a0_vals, t_vals)
    fig, ax = plt.subplots(dpi=300)
    ax.plot(t_vals, dA_dT, '-o', color='blue')
    ax.set_xlabel(r'$T$')
    ax.set_ylabel(r'$dA_{\mathrm{AN}}(0)/dT$')
    plt.show()


In [ ]:
# fig, ax = plt.subplots(dpi=300)

# contrast = (results["peak_XG"] - results["peak_MX"]) / (results["peak_XG"] + results["peak_MX"])
# contrast_err = np.sqrt( (results["err_XG"]**2 + results["err_MX"]**2) ) / (results["peak_XG"] + results["peak_MX"])

# ax.errorbar(results["T"], contrast, yerr=contrast_err, 
#             fmt='-o', color='black')

# ax.axvline(x=Tc, color='gray', linestyle=':', linewidth=1.5, label=rf'$T_c$')
# ax.axhline(y=0, color='black', linestyle='--', linewidth=1)

# ax.set_xlabel(r'$T$')
# ax.set_ylabel('Fermi surface contrast')
# ax.set_xlim(0,0.2) 
# # ax.set_ylim(-0.01,0.21) 
# ax.legend(loc='best', frameon=False)
# plt.show()